<a href="https://colab.research.google.com/github/tobiartinian/Econom-a-y-Finanzas/blob/main/EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
%pip install jupysql
%pip install duckdb-engine
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False
%sql duckdb://

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.8/662.8 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.7 MB/s eta 0:00:00


In [2]:
dataset_path = "/content/drive/MyDrive/Economía y finanzas/"
dataset_file = "competencia_01.csv"

##EDA

In [3]:
%%sql
create or replace table competencia_01 as
select
    *
from read_csv_auto("{{dataset_path + dataset_file}}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Success


In [5]:
%%sql
select DISTINCT(foto_mes)
from competencia_01
order by foto_mes ASC
# tenemos info desde marzo 21 a agosto 21

,foto_mes
0,202103
1,202104
2,202105
3,202106
4,202107
5,202108


In [ ]:
%%sql
SELECT
AVG(active_quarter) AS tasa_activacion,
foto_mes
FROM competencia_01
GROUP BY foto_mes


,tasa_activacion,foto_mes
0,0.985734,202103
1,0.986876,202104
2,0.987275,202105
3,0.987722,202106
4,0.987478,202107
5,0.986632,202108


In [ ]:
%%sql
SELECT
clase_ternaria,
AVG(active_quarter) as tasa_activacion
FROM competencia_01
GROUP BY clase_Ternaria
ORDER BY tasa_activacion DESC

,clase_ternaria,tasa_activacion
0,CONTINUA,0.988621
1,None,0.987453
2,BAJA+2,0.859848
3,BAJA+1,0.844733


In [ ]:
%%sql
SELECT
    clase_ternaria,
    COUNT(*) AS cantidad_vip,
    PRINTF('%.2f%%', COUNT(*) * 100.0 / SUM(COUNT(*)) OVER()) AS porcentaje_vip
FROM competencia_01
WHERE cliente_vip = 1
GROUP BY clase_ternaria
ORDER BY cantidad_vip DESC;

,clase_ternaria,cantidad_vip,porcentaje_vip
0,CONTINUA,2200,70.69%
1,None,905,29.08%
2,BAJA+1,4,0.13%
3,BAJA+2,3,0.10%


In [ ]:
%%sql
SELECT
  PRINTF('%.2f%%',AVG(internet)*100) as tasa_internet
FROM competencia_01

,tasa_internet
0,4.91%


In [8]:
%%sql
SELECT
  clase_ternaria,
  PRINTF('%.2f%%',AVG(internet)*100) as tasa_internet
FROM competencia_01
GROUP BY clase_ternaria

,clase_ternaria,tasa_internet
0,None,3.50%
1,BAJA+2,20.19%
2,CONTINUA,5.47%
3,BAJA+1,13.09%


In [11]:
%%sql
SELECT
AVG(cliente_edad) as avg_edad,
MAX(cliente_edad) as max_edad,
MIN(cliente_edad) as min_edad,
PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY cliente_edad) AS mediana_edad
FROM competencia_01

,avg_edad,max_edad,min_edad,mediana_edad
0,46.884686,101,18,45.0


In [28]:
%%sql
WITH base AS (
    SELECT
        numero_de_cliente,
        clase_ternaria,
        CASE
            WHEN cliente_edad < 30 THEN '1. Menor a 30'
            WHEN cliente_edad < 40 THEN '2. 30 a 39'
            WHEN cliente_edad < 50 THEN '3. 40 a 49'
            WHEN cliente_edad < 60 THEN '4. 50 a 59'
            ELSE '5. 60 o más'
        END AS rango_edad
    FROM competencia_01
)
SELECT
    rango_edad,
    clase_ternaria,
    -- Porcentaje sobre el TOTAL GLOBAL de la tabla
    ROUND(100.0 * COUNT(DISTINCT numero_de_cliente) / SUM(COUNT(DISTINCT numero_de_cliente)) OVER(), 2) AS pct_sobre_total_global,
    ROUND(100.0 * COUNT(DISTINCT numero_de_cliente) / SUM(COUNT(DISTINCT numero_de_cliente)) OVER(PARTITION BY rango_edad), 2) AS pct_sobre_total_rango,
FROM base
GROUP BY rango_edad, clase_ternaria
ORDER BY rango_edad, clase_ternaria;

,rango_edad,clase_ternaria,pct_sobre_total_global,pct_sobre_total_rango
0,1. Menor a 30,BAJA+1,0.11,1.50
1,1. Menor a 30,BAJA+2,0.09,1.24
2,1. Menor a 30,CONTINUA,3.55,49.97
3,1. Menor a 30,None,3.36,47.29
4,2. 30 a 39,BAJA+1,0.37,1.45
5,2. 30 a 39,BAJA+2,0.31,1.19
6,2. 30 a 39,CONTINUA,12.73,49.36
7,2. 30 a 39,None,12.37,47.99
8,3. 40 a 49,BAJA+1,0.39,1.33
9,3. 40 a 49,BAJA+2,0.30,1.04
